# Affichage d'une carte de température de la consommation d'énergie par région

In [10]:
from urllib.request import urlopen
import os
import json
import pandas as pd
import plotly.express as px

In [11]:
with open("../data/regions.geojson","r") as reg_geo:
    regions = json.load(reg_geo)

In [12]:
from mix_energy import eco2mix_ingest as eco

def build_df(dataset:str)->pd.DataFrame:
    print("Build a DataFrame instance from the dataset {}".format(dataset))
    
    dataset_file = os.path.join("../data/",f"{dataset}.csv")
    
    if not os.path.exists(dataset_file) :
        csv_content = eco.retrieve_csv(dataset_id=dataset)
        with open(dataset_file,"wb") as csv_file:
            csv_file.write(csv_content)
            
    df = pd.read_csv(dataset_file,sep=";")
    
    return df

df = pd.read_csv("../data/eco2mix-regional-tr.csv",sep=';')
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 74880 entries, 0 to 74879
Data columns (total 30 columns):
 #   Column               Non-Null Count  Dtype  
---  ------               --------------  -----  
 0   Code INSEE région    74880 non-null  int64  
 1   Région               74880 non-null  str    
 2   Nature               74880 non-null  str    
 3   Date                 74880 non-null  str    
 4   Heure                74880 non-null  str    
 5   Date - Heure         74880 non-null  str    
 6   Consommation (MW)    74736 non-null  float64
 7   Thermique (MW)       74736 non-null  float64
 8   Nucléaire (MW)       74736 non-null  float64
 9   Eolien (MW)          74736 non-null  float64
 10  Solaire (MW)         74736 non-null  float64
 11  Hydraulique (MW)     74736 non-null  float64
 12  Pompage (MW)         74736 non-null  float64
 13  Bioénergies (MW)     74736 non-null  float64
 14  Ech. physiques (MW)  74736 non-null  float64
 15  Stockage batterie    74736 non-null  float64
 1

In [ ]:
df.dropna()
df = df.drop(columns=["column_68"])
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 74880 entries, 0 to 74879
Data columns (total 29 columns):
 #   Column               Non-Null Count  Dtype  
---  ------               --------------  -----  
 0   Code INSEE région    74880 non-null  int64  
 1   Région               74880 non-null  str    
 2   Nature               74880 non-null  str    
 3   Date                 74880 non-null  str    
 4   Heure                74880 non-null  str    
 5   Date - Heure         74880 non-null  str    
 6   Consommation (MW)    74736 non-null  float64
 7   Thermique (MW)       74736 non-null  float64
 8   Nucléaire (MW)       74736 non-null  float64
 9   Eolien (MW)          74736 non-null  float64
 10  Solaire (MW)         74736 non-null  float64
 11  Hydraulique (MW)     74736 non-null  float64
 12  Pompage (MW)         74736 non-null  float64
 13  Bioénergies (MW)     74736 non-null  float64
 14  Ech. physiques (MW)  74736 non-null  float64
 15  Stockage batterie    74736 non-null  float64
 1

In [ ]:
df_reduced = df.groupby(by=["code_insee_region","date","libelle_region"],as_index=False)["consommation"].sum()
df_reduced["date"].max()
df_reduced = df_reduced[df_reduced.date==df_reduced.date.max()]
df_last = df_reduced.drop(columns=["date"])




In [ ]:
df_last = df_last.rename(columns={"code_insee_region":"code"})
df_last.head()

In [ ]:
df_last.describe()

In [ ]:
reg_df = pd.json_normalize(regions["features"])
reg_df = reg_df.rename(columns={"properties.code":"code"}) 
reg_df.head()

In [5]:
import plotly.express as px

list_reg = list(df_last["libelle_region"].unique())

fig = px.choropleth_map(df_last, geojson=regions, 
                        locations='code', 
                        color='consommation',
                        featureidkey="properties.code",
                        color_continuous_scale="Hot",
                        range_color=(50000, 500000),
                        map_style="carto-positron",
                        zoom=4, 
                        #center = {"lat": 43.327408, "lon": -1.032999}, Saint-Palais
                        #center = {"lat": 48.866667, "lon": 2.333333}, Paris
                        center = {"lat": 47.0, "lon": 1.909000}, # Autour d'Orléans (lon: 47.902500, lat: 1.909000)
                        opacity=0.5,
                        hover_data=["libelle_region","consommation"],
                        labels={'consommation':'consommation energie'},
                        )
fig.update_layout(margin={"r":0,"t":0,"l":0,"b":0})
fig.show()

NameError: name 'df_last' is not defined